# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EarlPly/Start-W1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane:** Structured Content Archetype Clustering. This is a deliberately simple editorial action baseline for that lane: a transparent list of pages to inspect before a later model can claim to improve triage. Data are the 30,000-row anonymized starter CSV, one row per pseudonymous content item, with trailing 90-day search metrics. This is decision support, not estimated intervention benefit.


## 1. Two signal checks, then the rule

**Signal 1 — CTR versus search position (FlyRank CTR-fix flag). Verdict: MIXED.** With at least 300 impressions and valid position, median CTR is 0.23% at positions 4–10, 0.17% at 11–20, and 0.08% at 21–50. The top-three group is only 0.20%, below the 4–10 group, so this is not a clean monotone pattern. Position is an aggregate; query mix and SERP features are unobserved. The table below gives every bucket's `n` and both median and impression-weighted CTR.

**Signal 2 — volume versus zero clicks (FlyRank quick-win volume flag). Verdict: CONFIRMED.** Among valid-position pages with at least 300 impressions, the zero-click share falls from 49.4% at 300–999 impressions to 14.6% at 1,000–4,999 and 1.3% at 5,000+. The count pattern is partly mechanical (more impressions mean more chances for a click); it supports a minimum evidence floor, not a causal quick-win claim. The table prints `n`.

**Plain-language rule:** Review a title/snippet when a page had at least **300 impressions**, averaged **positions >3 through 10**, and recorded fewer clicks than it would have at the cohort's **0.23% median CTR**. Rank by the approximate click gap: `max(0, 0.0023 × impressions_90d − clicks_90d)`. The 0.23% reference is frozen here after this signal check; it is not a fitted predictive model. The single reason code is `LOW_CTR_PAGE1`, and the single action label is `Review title/snippet`. An editor must inspect the actual page, queries, and SERP before making any change.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

# A local checkout uses its shipped starter CSV. Colab can read the same public,
# anonymized file from GitHub without a warehouse token.
local_paths = [Path('data/raw/content_refresh_anonymized.csv'),
               Path('../../data/raw/content_refresh_anonymized.csv')]
source = next((str(p) for p in local_paths if p.exists()),
              'https://raw.githubusercontent.com/EarlPly/Start-W1/main/data/raw/content_refresh_anonymized.csv')
pages = pd.read_csv(source)
required = {'content_id', 'impressions_90d', 'clicks_90d', 'ctr', 'avg_position'}
assert required <= set(pages.columns)
assert len(pages) == 30_000 and pages.content_id.is_unique
valid = pages.loc[(pages.impressions_90d >= 300) & (pages.avg_position > 0)].copy()

def bucket_table(frame, group):
    out = frame.groupby(group, observed=True).apply(
        lambda g: pd.Series({
            'n': len(g),
            'median_ctr_pct': g.ctr.median(),
            'weighted_ctr_pct': 100 * g.clicks_90d.sum() / g.impressions_90d.sum(),
            'zero_click_pct': 100 * (g.clicks_90d == 0).mean(),
        }), include_groups=False)
    out['n'] = out['n'].astype(int)
    return out.round(3)

valid['position_bucket'] = pd.cut(valid.avg_position, [0, 3, 10, 20, 50, np.inf],
                                  labels=['top_3', '4-10', '11-20', '21-50', '51+'])
valid['volume_bucket'] = pd.cut(valid.impressions_90d, [299, 999, 4999, np.inf],
                                labels=['300-999', '1k-4.9k', '5k+'])
position_signal = bucket_table(valid, 'position_bucket')
volume_signal = bucket_table(valid, 'volume_bucket')
print(f'Starter rows: {len(pages):,}; valid position plus >=300 impressions: {len(valid):,}')
print('Signal 1 — CTR by position; verdict MIXED')
display(position_signal)
print('Signal 2 — zero clicks by volume; verdict CONFIRMED')
display(volume_signal)
assert position_signal.loc['4-10', 'n'] > 0
assert round(position_signal.loc['4-10', 'median_ctr_pct'], 2) == 0.23


Starter rows: 30,000; valid position plus >=300 impressions: 18,752
Signal 1 — CTR by position; verdict MIXED
Signal 2 — zero clicks by volume; verdict CONFIRMED


,n,median_ctr_pct,weighted_ctr_pct,zero_click_pct
position_bucket,,,,
top_3,507,0.20,0.489,14.793
4-10,7647,0.23,0.348,9.873
11-20,5051,0.17,0.349,19.699
21-50,4985,0.08,0.154,30.110
51+,562,0.00,0.037,73.310


,n,median_ctr_pct,weighted_ctr_pct,zero_click_pct
volume_bucket,,,,
300-999,5240,0.10,0.195,49.408
1k-4.9k,7361,0.16,0.260,14.563
5k+,6151,0.22,0.319,1.252


## 2. Build the ranked queue

The rule uses only trailing-90-day impressions, clicks, and average position. `avg_position = 0` means missing and is ineligible. The score is an **approximate gap against a peer reference**, not a forecast of clicks gained from an edit. The CSV contains pseudonymous IDs and aggregate measurements only. Zero-score and ineligible pages are not action items.


In [2]:
REFERENCE_CTR_PCT = 0.23
REASON_CODE = 'LOW_CTR_PAGE1'
ACTION = 'Review title/snippet'

eligible = pages.loc[(pages.impressions_90d >= 300)
                     & (pages.avg_position > 3)
                     & (pages.avg_position <= 10)].copy()
eligible['actual_ctr_pct'] = 100 * eligible.clicks_90d / eligible.impressions_90d
eligible['score'] = (REFERENCE_CTR_PCT / 100 * eligible.impressions_90d
                     - eligible.clicks_90d).clip(lower=0)
queue = eligible.loc[eligible.score > 0, [
    'content_id', 'impressions_90d', 'clicks_90d', 'actual_ctr_pct',
    'avg_position', 'score']].copy()
queue = queue.sort_values(['score', 'impressions_90d', 'content_id'],
                          ascending=[False, False, True]).reset_index(drop=True)
queue.insert(0, 'rank', np.arange(1, len(queue) + 1))
queue['reason_code'] = REASON_CODE
queue['action_label'] = ACTION
queue['reference_ctr_pct'] = REFERENCE_CTR_PCT
out_dir = Path('work/outputs')
out_dir.mkdir(parents=True, exist_ok=True)
queue_path = out_dir / 'baseline_action_score.csv'
queue.to_csv(queue_path, index=False)
assert queue.content_id.is_unique and queue.score.is_monotonic_decreasing
assert queue.reason_code.nunique() == queue.action_label.nunique() == 1
assert set(queue.columns).isdisjoint({'trend_pct', 'trend_direction', 'is_declining_label'})
print(f'Eligible pages: {len(eligible):,}; positive-score queue: {len(queue):,}')
print(f'Wrote {queue_path} ({len(queue):,} ranked rows).')
display(queue.head(10).round({'actual_ctr_pct': 3, 'score': 1}))


Eligible pages: 7,647; positive-score queue: 3,756
Wrote work/outputs/baseline_action_score.csv (3,756 ranked rows).


,rank,content_id,impressions_90d,clicks_90d,actual_ctr_pct,avg_position,score,reason_code,action_label,reference_ctr_pct
0,1,content_36ff89c8214e,295097,154,0.052,7.3,524.7,LOW_CTR_PAGE1,Review title/snippet,0.23
1,2,content_c8e9d6ab9013,208678,0,0.000,9.7,480.0,LOW_CTR_PAGE1,Review title/snippet,0.23
2,3,content_5fe46e04994d,517715,741,0.143,4.2,449.7,LOW_CTR_PAGE1,Review title/snippet,0.23
3,4,content_c84a0ab98e90,223271,70,0.031,7.8,443.5,LOW_CTR_PAGE1,Review title/snippet,0.23
4,5,content_453722754fea,140079,16,0.011,7.6,306.2,LOW_CTR_PAGE1,Review title/snippet,0.23
5,6,content_73c54f78c06a,213963,211,0.099,4.7,281.1,LOW_CTR_PAGE1,Review title/snippet,0.23
6,7,content_91652435f57a,159590,100,0.063,7.8,267.1,LOW_CTR_PAGE1,Review title/snippet,0.23
7,8,content_c1fe78bc4e37,134055,43,0.032,7.5,265.3,LOW_CTR_PAGE1,Review title/snippet,0.23
8,9,content_39881853ef0c,112434,10,0.009,7.2,248.6,LOW_CTR_PAGE1,Review title/snippet,0.23
9,10,content_0919dd345d80,119217,26,0.022,7.0,248.2,LOW_CTR_PAGE1,Review title/snippet,0.23


## 3. Top-20 review (top ten are the assignment decision list)

Each line below gives the proposed action, why its score is high, a confidence note, and one concrete reason the pick could be wrong. All twenty are **review candidates only**. The first ten are the requested top-ten review; the next ten extend the skeleton's top-twenty check. Pseudonyms and aggregate metrics cannot reveal query intent or the rendered search result, so none is an automatic edit recommendation.


In [3]:
# Deliberately reviewed caveats keyed to the observed top-20 IDs.
caveats = {
 'content_36ff89c8214e': 'Query mix or SERP features may explain the gap despite position 7.3.',
 'content_c8e9d6ab9013': 'Zero clicks from 208,678 impressions could indicate measurement or indexing trouble.',
 'content_5fe46e04994d': 'Its very large volume can dominate the score; 0.14% CTR may fit its query intent.',
 'content_c84a0ab98e90': 'A recent update means the snippet may already have changed during this 90-day window.',
 'content_453722754fea': 'Sixteen clicks on high volume might reflect a branded or answer-on-SERP query.',
 'content_73c54f78c06a': 'Position 4.7 is an average; query-level positions may differ sharply.',
 'content_91652435f57a': 'Commercial intent may have a different CTR benchmark than the pooled cohort.',
 'content_c1fe78bc4e37': 'A commercial page may face ads or shopping results that depress organic CTR.',
 'content_39881853ef0c': 'A recent update may not yet have a full post-edit measurement window.',
 'content_0919dd345d80': 'Its seven-day-old update makes the 90-day average slow to reflect the current snippet.',
 'content_b115f7c74779': 'Transactional SERPs can have paid blocks that make the pooled reference too high.',
 'content_a7427266c305': 'Position 5.7 and query mix may explain a naturally lower CTR.',
 'content_97a86caf3a3d': 'Transactional intent could make this cohort-wide CTR reference unsuitable.',
 'content_cb112fce36be': 'Already 492 clicks; a title change could harm a working query segment.',
 'content_4c76e9b13aea': 'Position 7.4 may be the main constraint rather than wording.',
 'content_63f88d16fdb8': 'Its lower volume makes the estimated gap less stable than the top five.',
 'content_9e08e86d0824': 'A recent update means the current search listing should be checked first.',
 'content_d0cc5baa4995': 'Transactional query mix may not match the pooled page-one median.',
 'content_f42eb861c6dd': 'CTR 0.13% may be normal for its query mix; inspect queries before editing.',
 'content_b6100a228edc': 'The recent update may have changed the live title during the observation window.',
}
top20 = queue.head(20).merge(
    pages[['content_id', 'main_intent', 'days_since_last_update']],
    on='content_id', validate='one_to_one')
assert set(top20.content_id) == set(caveats), 'Review the new top 20 if the ranking changes.'
top20['why_there'] = top20.apply(
    lambda r: (f"{int(r.impressions_90d):,} impressions, position {r.avg_position:.1f}, "
               f"CTR {r.actual_ctr_pct:.2f}% vs {REFERENCE_CTR_PCT:.2f}% reference; "
               f"gap {r.score:.0f} clicks"), axis=1)
top20['confidence_note'] = top20.apply(
    lambda r: 'Low: zero observed clicks; verify tracking' if r.clicks_90d == 0
    else 'Provisional: aggregate search data only', axis=1)
top20['what_would_make_it_wrong'] = top20.content_id.map(caveats)
review = top20[['rank', 'content_id', 'main_intent', 'days_since_last_update',
                'action_label', 'reason_code', 'why_there',
                'confidence_note', 'what_would_make_it_wrong']]
print('Top ten requested review; rows 11–20 extend the notebook skeleton.')
display(review)


Top ten requested review; rows 11–20 extend the notebook skeleton.


,rank,content_id,main_intent,days_since_last_update,action_label,reason_code,why_there,confidence_note,what_would_make_it_wrong
0,1,content_36ff89c8214e,informational,104,Review title/snippet,LOW_CTR_PAGE1,"295,097 impressions, position 7.3, CTR 0.05% v...",Provisional: aggregate search data only,Query mix or SERP features may explain the gap...
1,2,content_c8e9d6ab9013,informational,104,Review title/snippet,LOW_CTR_PAGE1,"208,678 impressions, position 9.7, CTR 0.00% v...",Low: zero observed clicks; verify tracking,"Zero clicks from 208,678 impressions could ind..."
2,3,content_5fe46e04994d,informational,104,Review title/snippet,LOW_CTR_PAGE1,"517,715 impressions, position 4.2, CTR 0.14% v...",Provisional: aggregate search data only,Its very large volume can dominate the score; ...
3,4,content_c84a0ab98e90,informational,20,Review title/snippet,LOW_CTR_PAGE1,"223,271 impressions, position 7.8, CTR 0.03% v...",Provisional: aggregate search data only,A recent update means the snippet may already ...
4,5,content_453722754fea,informational,20,Review title/snippet,LOW_CTR_PAGE1,"140,079 impressions, position 7.6, CTR 0.01% v...",Provisional: aggregate search data only,Sixteen clicks on high volume might reflect a ...
5,6,content_73c54f78c06a,informational,20,Review title/snippet,LOW_CTR_PAGE1,"213,963 impressions, position 4.7, CTR 0.10% v...",Provisional: aggregate search data only,Position 4.7 is an average; query-level positi...
6,7,content_91652435f57a,commercial,104,Review title/snippet,LOW_CTR_PAGE1,"159,590 impressions, position 7.8, CTR 0.06% v...",Provisional: aggregate search data only,Commercial intent may have a different CTR ben...
7,8,content_c1fe78bc4e37,commercial,104,Review title/snippet,LOW_CTR_PAGE1,"134,055 impressions, position 7.5, CTR 0.03% v...",Provisional: aggregate search data only,A commercial page may face ads or shopping res...
8,9,content_39881853ef0c,informational,20,Review title/snippet,LOW_CTR_PAGE1,"112,434 impressions, position 7.2, CTR 0.01% v...",Provisional: aggregate search data only,A recent update may not yet have a full post-e...
9,10,content_0919dd345d80,informational,7,Review title/snippet,LOW_CTR_PAGE1,"119,217 impressions, position 7.0, CTR 0.02% v...",Provisional: aggregate search data only,Its seven-day-old update makes the 90-day aver...


## 4. Weak picks and leakage check

The zero-click second-ranked page is a **weak recommendation for an immediate edit** until tracking and indexing are checked; the ninth and tenth picks had recent updates, so their 90-day average may be stale. A small positive score near the bottom of the queue is also a weak operational priority. These are the reasons to keep the action at *review*.

There is no independent editorial success label in this starter slice. `trend_direction` is derived from `trend_pct`, and the latter is built from recent comparison windows; using either as an input or claiming precision@K against it would create a misleading evaluation. This baseline has an auditable top-K review, but **no valid precision@K claim** until independently judged outcomes are available. The CSV is deliberately ignored by git under the repo's dataset/output rules; running this notebook creates it.


In [4]:
weak = queue.tail(5)[['rank', 'content_id', 'impressions_90d',
                      'clicks_90d', 'avg_position', 'score']]
print('Five lowest positive-score queue items (weak priority):')
display(weak.round({'score': 3}))
for forbidden in ['trend_pct', 'trend_direction', 'is_declining_label',
                  'impressions_last_30d', 'impressions_prev_30d']:
    assert forbidden not in queue.columns
assert (queue.impressions_90d >= 300).all()
assert queue.avg_position.gt(3).all() and queue.avg_position.le(10).all()
assert (queue.actual_ctr_pct < REFERENCE_CTR_PCT).all()
assert len(pd.read_csv(queue_path)) == len(queue)
print('Leakage and queue checks passed: no label-derived/recent-window inputs, valid position and volume, one reason/action.')
print('Precision@K: not claimed; no independent editorial ground truth in the starter CSV.')


Five lowest positive-score queue items (weak priority):
Leakage and queue checks passed: no label-derived/recent-window inputs, valid position and volume, one reason/action.
Precision@K: not claimed; no independent editorial ground truth in the starter CSV.


,rank,content_id,impressions_90d,clicks_90d,avg_position,score
3751,3752,content_86a178a6d1f1,440,1,8.1,0.012
3752,3753,content_60528f64fe02,2179,5,6.2,0.012
3753,3754,content_b27487ab3b2d,439,1,6.0,0.010
3754,3755,content_07d062db2b51,873,2,8.1,0.008
3755,3756,content_4f1f8c47f2a7,2176,5,5.3,0.005


## 5. Self-check

- [x] Two signal bucket tables print `n` and one-word verdicts; the CTR-position and volume checks are tied to FlyRank product flags.
- [x] One plain-language rule has a transparent score, one reason code, and one action label.
- [x] The notebook writes `work/outputs/baseline_action_score.csv` and shows a ranked queue.
- [x] The top ten each have action, reason, confidence, and a failure mode; rows 11–20 extend the review.
- [x] Weak picks and missing ground truth are stated; no future-window or label-derived inputs enter the score.
- [x] Executed top to bottom with pseudonymous IDs only, then committed under `work/notebooks/`.
